In [4]:
import os
os.listdir("data_processed")

['weather_lga_2022.csv',
 'daily_citibike_2022.csv',
 'citibike_weather_daily_2022.csv']

In [2]:
Path = ("/Users/mehreenwerth/Desktop/citibike-weather-2022")

In [7]:
import pandas as pd
trips_raw = pd.read_csv("data_processed/citibike_weather_daily_2022.csv", low_memory=False)
trips_raw.shape, trips_raw.columns

((365, 30),
 Index(['date', 'trip_count', 'avg_trip_minutes', 'member_trips',
        'casual_trips', 'ADPT', 'ASLP', 'ASTP', 'AWBT', 'AWND', 'PRCP', 'RHAV',
        'RHMN', 'RHMX', 'SNOW', 'SNWD', 'TAVG', 'TMAX', 'TMIN', 'WDF2', 'WDF5',
        'WSF2', 'WSF5', 'WT01', 'WT02', 'WT03', 'WT04', 'WT06', 'WT08', 'WT09'],
       dtype='object'))

In [10]:
from pathlib import Path

RAW_DIR = Path("/Users/mehreenwerth/Desktop/citibike-weather-2022/data_raw/csv")

files = sorted(list(RAW_DIR.glob("*.csv")))
len(files), files[:5]

(36,
 [PosixPath('/Users/mehreenwerth/Desktop/citibike-weather-2022/data_raw/csv/202201-citibike-tripdata_1.csv'),
  PosixPath('/Users/mehreenwerth/Desktop/citibike-weather-2022/data_raw/csv/202201-citibike-tripdata_2.csv'),
  PosixPath('/Users/mehreenwerth/Desktop/citibike-weather-2022/data_raw/csv/202202-citibike-tripdata_1.csv'),
  PosixPath('/Users/mehreenwerth/Desktop/citibike-weather-2022/data_raw/csv/202202-citibike-tripdata_2.csv'),
  PosixPath('/Users/mehreenwerth/Desktop/citibike-weather-2022/data_raw/csv/202203-citibike-tripdata_1.csv')])

In [11]:
import pandas as pd
import numpy as np

files = sorted(list(RAW_DIR.glob("*.csv")))
if not files:
    files = sorted(list(RAW_DIR.glob("*.csv.gz")))

def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Citi Bike files sometimes use different column naming styles.
    This function renames common variants into standard names.
    """
    rename_map = {}

    if "start_station_name" not in df.columns:
        for cand in ["Start Station Name", "start station name", "start_station", "startStationName"]:
            if cand in df.columns:
                rename_map[cand] = "start_station_name"
                break

    if "member_casual" not in df.columns:
        for cand in ["Member Type", "usertype", "member type", "rideable_type"]:  # rideable_type is NOT member/casual, but keep for later
            if cand in df.columns:
                # Only map if it actually represents member/casual; many older datasets use "usertype"
                if cand.lower() in ["usertype", "member type", "member type".lower()]:
                    rename_map[cand] = "member_casual"
                break

    if "started_at" not in df.columns:
        for cand in ["started_at", "Start Time", "starttime", "start_time"]:
            if cand in df.columns:
                rename_map[cand] = "started_at"
                break

    if "ended_at" not in df.columns:
        for cand in ["ended_at", "Stop Time", "stoptime", "stop_time"]:
            if cand in df.columns:
                rename_map[cand] = "ended_at"
                break

    if rename_map:
        df = df.rename(columns=rename_map)

    return df


desired_cols = ["started_at", "ended_at", "start_station_name", "member_casual"]

chunks = []
for f in files:
    print("Reading:", f.name)
    temp = pd.read_csv(f, low_memory=False)

    temp = standardize_columns(temp)

    keep = [c for c in desired_cols if c in temp.columns]
    temp = temp[keep].copy()

    if "started_at" in temp.columns:
        temp["started_at"] = pd.to_datetime(temp["started_at"], errors="coerce")
    if "ended_at" in temp.columns:
        temp["ended_at"] = pd.to_datetime(temp["ended_at"], errors="coerce")

        chunks.append(temp)

master_trips = pd.concat(chunks, ignore_index=True)
master_trips.shape, master_trips.head()


Reading: 202201-citibike-tripdata_1.csv
Reading: 202201-citibike-tripdata_2.csv
Reading: 202202-citibike-tripdata_1.csv
Reading: 202202-citibike-tripdata_2.csv
Reading: 202203-citibike-tripdata_1.csv
Reading: 202203-citibike-tripdata_2.csv
Reading: 202204-citibike-tripdata_1.csv
Reading: 202204-citibike-tripdata_2.csv
Reading: 202204-citibike-tripdata_3.csv
Reading: 202205-citibike-tripdata_1.csv
Reading: 202205-citibike-tripdata_2.csv
Reading: 202205-citibike-tripdata_3.csv
Reading: 202206-citibike-tripdata_1.csv
Reading: 202206-citibike-tripdata_2.csv
Reading: 202206-citibike-tripdata_3.csv
Reading: 202206-citibike-tripdata_4.csv
Reading: 202207-citibike-tripdata_1.csv
Reading: 202207-citibike-tripdata_2.csv
Reading: 202207-citibike-tripdata_3.csv
Reading: 202207-citibike-tripdata_4.csv
Reading: 202208-citibike-tripdata_1.csv
Reading: 202208-citibike-tripdata_2.csv
Reading: 202208-citibike-tripdata_3.csv
Reading: 202208-citibike-tripdata_4.csv
Reading: 202209-citibike-tripdata_1.csv


((29838806, 4),
                started_at                ended_at       start_station_name  \
 0 2022-01-21 13:13:43.392 2022-01-21 13:22:31.463  West End Ave & W 107 St   
 1 2022-01-10 11:30:54.162 2022-01-10 11:41:43.422             4 Ave & 3 St   
 2 2022-01-26 10:52:43.096 2022-01-26 11:06:35.227          1 Ave & E 62 St   
 3 2022-01-03 08:35:48.247 2022-01-03 09:10:50.475          2 Ave & E 96 St   
 4 2022-01-22 14:14:23.043 2022-01-22 14:34:57.474          6 Ave & W 34 St   
 
   member_casual  
 0        member  
 1        member  
 2        member  
 3        member  
 4        member  )

In [13]:
OUT_DIR = Path("/Users/mehreenwerth/Desktop/citibike-weather-2022/data_processed")
out_path = OUT_DIR / "citibike_master_2022.csv"

master_trips.to_csv(out_path, index=False)

out_path

PosixPath('/Users/mehreenwerth/Desktop/citibike-weather-2022/data_processed/citibike_master_2022.csv')